# LeetCode 1009 — Complement of Base 10 Integer bằng Python

**Link đề:** https://leetcode.com/problems/complement-of-base-10-integer/
**Độ khó:** Easy · **Chủ đề:** Bit Manipulation (thao tác bit) · Giống hệt bài 476 (Number Complement)

```python
class Solution:
    def bitwiseComplement(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **biểu diễn nhị phân** (binary) của một số nguyên và cách đổi qua lại hệ 10 ↔ hệ 2.
2. Làm quen các công cụ Python: `bin`, `int(s, 2)`, `format`, `%`, `//`, `<<`, `>>`, `&`, `|`, `^`, `bit_length`.
3. Viết lời giải trực giác bằng **chuỗi** rồi bằng **vòng lặp từng bit**.
4. Rút ra insight: `n + complement = 11...1` (toàn bit 1) → `complement = mask - n = mask ^ n`.
5. Xử lý **edge case `n = 0`** (bẫy kinh điển của bài).
6. Trace, chứng minh, phân tích độ phức tạp, so sánh 4 cách, làm bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — lật mọi bit trong biểu diễn nhị phân

**Complement** (phần bù) của một số nguyên là số thu được khi lật mọi bit `0 → 1` và `1 → 0` trong biểu diễn nhị phân **không có số 0 thừa ở đầu** của nó.

Cho số nguyên `n`, trả về complement của `n`.

| `n` | Nhị phân | Lật bit | Output |
|---:|:---:|:---:|---:|
| 5 | `101` | `010` | 2 |
| 7 | `111` | `000` | 0 |
| 10 | `1010` | `0101` | 5 |
| 0 | `0` | `1` | **1** |

**Constraints:** `0 <= n < 10^9`.

Câu hỏi tự đặt:
- "Lật mọi bit" là lật bao nhiêu bit? (Chỉ các bit trong biểu diễn ngắn nhất: `5 = 101` có 3 bit, không phải 32 bit.)
- `n = 0` viết là `"0"` — một bit → lật thành `"1"` → đáp án `1`. (Gợi ý của đề cũng nhắc: *N = 0 là corner case*.)

In [1]:
# Xem nhị phân của các ví dụ
for n in [5, 7, 10, 0]:
    print(n, "->", bin(n))

5 -> 0b101
7 -> 0b111
10 -> 0b1010
0 -> 0b0


`bin(n)` trả về chuỗi bắt đầu bằng tiền tố `'0b'` (báo hiệu "đây là nhị phân"). Ta sẽ bỏ tiền tố bằng slicing `[2:]` ở mục 2.

## 2. Nền tảng: hệ nhị phân và công cụ Python

### 2.1. Hệ nhị phân là gì?

Hệ 10 dùng các chữ số `0..9`, mỗi vị trí mạnh gấp 10 lần vị trí bên phải. Hệ 2 chỉ dùng `0, 1`, mỗi vị trí mạnh gấp **2** lần.

`1010₂ = 1·2³ + 0·2² + 1·2¹ + 0·2⁰ = 8 + 0 + 2 + 0 = 10`.

In [2]:
bits = "1010"
total = 0
for pos, ch in enumerate(reversed(bits)):   # pos = 0 là bit phải nhất
    value = int(ch) * 2 ** pos
    print(f"bit '{ch}' ở vị trí {pos} đóng góp {int(ch)} * 2^{pos} = {value}")
    total += value
print("Tổng:", total)

bit '0' ở vị trí 0 đóng góp 0 * 2^0 = 0
bit '1' ở vị trí 1 đóng góp 1 * 2^1 = 2
bit '0' ở vị trí 2 đóng góp 0 * 2^2 = 0
bit '1' ở vị trí 3 đóng góp 1 * 2^3 = 8
Tổng: 10


- `reversed(bits)` duyệt chuỗi từ phải sang trái.
- `enumerate(...)` cho thêm chỉ số đếm `pos` bắt đầu từ 0.
- `2 ** pos` là lũy thừa.

### 2.2. Đổi qua lại: `bin`, `int(s, 2)`, `format(n, 'b')`

In [3]:
n = 10
s = bin(n)
print(s, "| bỏ '0b':", s[2:])
print("format(n, 'b'):", format(n, "b"))      # không có tiền tố
print("int('1010', 2):", int("1010", 2))       # chuỗi nhị phân -> số
print("int('0101', 2):", int("0101", 2))       # số 0 ở đầu không sao

0b1010 | bỏ '0b': 1010
format(n, 'b'): 1010
int('1010', 2): 10
int('0101', 2): 5


### 2.3. Lấy từng bit bằng `% 2` và `// 2`

Giống `% 10`, `// 10` để lấy chữ số thập phân cuối, ở hệ 2 ta dùng:
- `n % 2` → bit cuối (phải nhất).
- `n // 2` → bỏ bit cuối.

In [4]:
n = 10
while n > 0:
    print(f"n = {n:>2} = {bin(n)[2:]:>4} | bit cuối n % 2 = {n % 2} | n // 2 = {n // 2}")
    n //= 2

n = 10 = 1010 | bit cuối n % 2 = 0 | n // 2 = 5
n =  5 =  101 | bit cuối n % 2 = 1 | n // 2 = 2
n =  2 =   10 | bit cuối n % 2 = 0 | n // 2 = 1
n =  1 =    1 | bit cuối n % 2 = 1 | n // 2 = 0


Đọc cột "bit cuối" từ dưới lên: `1, 0, 1, 0` → `1010`. Đúng nhị phân của 10.

### 2.4. Toán tử bit: `&`, `|`, `^`, `~`, `<<`, `>>`

| Toán tử | Tên | Ý nghĩa trên từng bit | Ví dụ (`5=101`, `3=011`) |
|---|---|---|---|
| `a & b` | AND | 1 nếu cả hai là 1 | `101 & 011 = 001` = 1 |
| `a \| b` | OR | 1 nếu ít nhất một là 1 | `101 \| 011 = 111` = 7 |
| `a ^ b` | XOR | 1 nếu **khác nhau** | `101 ^ 011 = 110` = 6 |
| `a << k` | dịch trái | nhân `2^k` | `5 << 1 = 1010` = 10 |
| `a >> k` | dịch phải | chia nguyên `2^k` | `5 >> 1 = 10` = 2 |
| `~a` | NOT | lật **mọi** bit (cả vô hạn bit 0 phía trước!) | `~5 = -6` |

In [5]:
a, b2 = 5, 3
print("a & b =", a & b2)
print("a | b =", a | b2)
print("a ^ b =", a ^ b2)
print("a << 1 =", a << 1, "| a >> 1 =", a >> 1)
print("~a =", ~a)

a & b = 1
a | b = 7
a ^ b = 6
a << 1 = 10 | a >> 1 = 2
~a = -6


### 2.5. Vì sao **không** dùng được `~n` trực tiếp?

Python coi số nguyên có **vô hạn bit** (bù hai): `5 = ...000101`. `~5` lật hết → `...111010` = `-6`. Ta chỉ muốn lật **3 bit cuối** → cần một **mặt nạ (mask)** để giữ đúng 3 bit đó.

### 2.6. XOR với 1 = lật bit

`x ^ 1` lật bit `x` (`0^1=1`, `1^1=0`); `x ^ 0` giữ nguyên. Vậy XOR với một mask toàn bit 1 sẽ lật đúng các bit nằm dưới mask.

In [6]:
for x in [0, 1]:
    print(f"{x} ^ 1 = {x ^ 1} | {x} ^ 0 = {x ^ 0}")
print("0b101 ^ 0b111 =", bin(0b101 ^ 0b111))

0 ^ 1 = 1 | 0 ^ 0 = 0
1 ^ 1 = 0 | 1 ^ 0 = 1
0b101 ^ 0b111 = 0b10


`0b101` là cách viết **literal nhị phân** trong Python (giống viết số `5`).

### 2.7. `bit_length()` — số bit cần để biểu diễn

In [7]:
for n in [0, 1, 5, 7, 8, 10, 10**9 - 1]:
    print(f"{n:>10} -> {bin(n)[2:]:>30} | bit_length = {n.bit_length()}")

         0 ->                              0 | bit_length = 0
         1 ->                              1 | bit_length = 1
         5 ->                            101 | bit_length = 3
         7 ->                            111 | bit_length = 3
         8 ->                           1000 | bit_length = 4
        10 ->                           1010 | bit_length = 4
 999999999 -> 111011100110101100100111111111 | bit_length = 30


Chú ý `(0).bit_length() == 0` — không phải 1! Đây chính là nguồn gốc của bẫy `n = 0`.

## 3. Cách trực giác — làm việc với chuỗi

### 3.1. Cách 1: đổi sang chuỗi nhị phân, lật từng ký tự, đổi lại

Ba bước:
1. `s = bin(n)[2:]` → chuỗi nhị phân.
2. Lật: `'1' → '0'`, `'0' → '1'`.
3. `int(flipped, 2)` → số.

In [8]:
def complement_string(n: int) -> int:
    s = bin(n)[2:]
    flipped = ""
    for ch in s:
        if ch == "1":
            flipped += "0"
        else:
            flipped += "1"
    return int(flipped, 2)

for n in [5, 7, 10, 0]:
    print(n, "->", complement_string(n))

5 -> 2
7 -> 0
10 -> 5
0 -> 1


Với `n = 0`: `bin(0)[2:] = "0"` → lật `"1"` → 1. Cách chuỗi **tự động** đúng edge case vì `bin(0)` vẫn có một ký tự.

### 3.2. Viết gọn bằng `join` và biểu thức điều kiện

- `"x" if cond else "y"` — biểu thức điều kiện một dòng.
- `"".join(iterable)` ghép các chuỗi con thành một chuỗi (nhanh hơn `+=` lặp lại).

In [9]:
print("".join(["a", "b", "c"]))
s = "1010"
print("".join("0" if ch == "1" else "1" for ch in s))

def complement_string_join(n: int) -> int:
    s = bin(n)[2:]
    return int("".join("0" if ch == "1" else "1" for ch in s), 2)

print([complement_string_join(n) for n in [5, 7, 10, 0]])

abc
0101
[2, 0, 5, 1]


Hoặc dùng `str.translate` với bảng đổi ký tự: `s.translate(str.maketrans("01", "10"))`.

In [10]:
table = str.maketrans("01", "10")
print("1010".translate(table))

0101


Code nộp LeetCode — cách chuỗi:

In [11]:
class SolutionString:
    def bitwiseComplement(self, n: int) -> int:
        s = bin(n)[2:]
        return int(s.translate(str.maketrans("01", "10")), 2)

assert [SolutionString().bitwiseComplement(n) for n in [5, 7, 10, 0]] == [2, 0, 5, 1]
print("SolutionString PASS")

SolutionString PASS


**Độ phức tạp:** gọi `d = số bit của n` (≈ `log₂ n`, tối đa 30 vì `n < 10^9 < 2^30`). Thời gian `O(d)`, bộ nhớ `O(d)` cho chuỗi.

## 4. Cách 2 — Lật từng bit bằng số học (không dùng chuỗi)

Duyệt bit từ phải sang trái bằng `% 2` và `// 2`. Bit `b` lật thành `1 - b`, đặt vào vị trí `pos` bằng cách cộng `(1 - b) * 2^pos`.

In [12]:
def complement_loop(n: int) -> int:
    if n == 0:
        return 1                      # vòng while dưới đây không chạy khi n = 0
    result = 0
    pos = 0
    while n > 0:
        bit = n % 2
        result += (1 - bit) << pos    # (1 - bit) * 2**pos
        n //= 2
        pos += 1
    return result

for n in [5, 7, 10, 0]:
    print(n, "->", complement_loop(n))

5 -> 2
7 -> 0
10 -> 5
0 -> 1


Ở cách này **phải** xử lý `n = 0` riêng: `while n > 0` không chạy lần nào → `result = 0`, sai (đáp án 1).

`(1 - bit) << pos` dùng dịch trái thay cho `* 2 ** pos` — cùng giá trị nhưng là cách viết quen thuộc của thao tác bit.

## 5. Nút thắt & insight: tổng của số và phần bù

Hai cách trên đều xử lý **từng bit một**. Có cách nào tính ngay một phát?

Gợi ý của đề: *"Một số nhị phân cộng phần bù của nó bằng `111...111`."*

```
   n          = 1 0 1 0   (10)
   complement = 0 1 0 1   ( 5)
   ---------------------
   tổng       = 1 1 1 1   (15)
```

Vì mỗi vị trí đúng một bên là 1 → không có nhớ (carry), tổng là toàn bit 1. Gọi `mask = 11...1` (cùng số bit với `n`), ta có:

- `complement = mask - n`
- hoặc `complement = mask ^ n` (XOR lật đúng các bit dưới mask, mục 2.6).

Còn `mask` có `d` bit 1 thì bằng `2^d - 1 = (1 << d) - 1`.

In [13]:
for n in [5, 7, 10]:
    d = n.bit_length()
    mask = (1 << d) - 1
    print(f"n={n:>2} ({bin(n)[2:]:>4}) d={d} mask={mask:>2} ({bin(mask)[2:]:>4}) "
          f"mask-n={mask - n} mask^n={mask ^ n}")

n= 5 ( 101) d=3 mask= 7 ( 111) mask-n=2 mask^n=2
n= 7 ( 111) d=3 mask= 7 ( 111) mask-n=0 mask^n=0
n=10 (1010) d=4 mask=15 (1111) mask-n=5 mask^n=5


### 5.1. Vì sao `(1 << d) - 1` là `d` bit 1?

`1 << d` = `1` theo sau bởi `d` số 0. Trừ 1 thì mượn liên tiếp → `d` số 1. Giống `1000 - 1 = 999` trong hệ 10.

In [14]:
for d in range(1, 6):
    print(f"d={d}: 1<<d = {bin(1 << d)[2:]:>6}, (1<<d)-1 = {bin((1 << d) - 1)[2:]:>6}")

d=1: 1<<d =     10, (1<<d)-1 =      1
d=2: 1<<d =    100, (1<<d)-1 =     11
d=3: 1<<d =   1000, (1<<d)-1 =    111
d=4: 1<<d =  10000, (1<<d)-1 =   1111
d=5: 1<<d = 100000, (1<<d)-1 =  11111


### 5.2. Bẫy `n = 0`

`(0).bit_length() = 0` → `mask = (1 << 0) - 1 = 0` → `0 ^ 0 = 0`. **Sai!** Đáp án phải là 1 (vì `0` viết là một bit `"0"`).

Sửa: dùng `max(1, n.bit_length())`, hoặc `if n == 0: return 1`.

In [15]:
n = 0
d = n.bit_length()
print("bit_length(0) =", d, "-> mask =", (1 << d) - 1, "-> sai:", ((1 << d) - 1) ^ n)
d = max(1, n.bit_length())
print("dùng max(1, ...) -> mask =", (1 << d) - 1, "-> đúng:", ((1 << d) - 1) ^ n)

bit_length(0) = 0 -> mask = 0 -> sai: 0
dùng max(1, ...) -> mask = 1 -> đúng: 1


## 6. Lời giải chính — mask bằng `bit_length` rồi XOR

Mổ xẻ:

```python
if n == 0:                       # Đoạn 1: edge case
    return 1
mask = (1 << n.bit_length()) - 1 # Đoạn 2: mask toàn bit 1, cùng độ dài với n
return n ^ mask                  # Đoạn 3: XOR lật đúng các bit đó
```

In [16]:
def complement_mask(n: int) -> int:
    if n == 0:
        return 1
    mask = (1 << n.bit_length()) - 1
    return n ^ mask

for n in [5, 7, 10, 0, 1, 8]:
    print(n, "->", complement_mask(n))

5 -> 2
7 -> 0
10 -> 5
0 -> 1
1 -> 0
8 -> 7


### 6.1. Cách 4 — tự xây mask bằng vòng lặp (khi không có `bit_length`)

Nhiều ngôn ngữ không có `bit_length`. Ta tự xây: bắt đầu `mask = 1`, chừng nào `mask < n` thì `mask = (mask << 1) | 1` (thêm một bit 1 bên phải).

Bắt đầu `mask = 1` cũng tự xử lý `n = 0`: `1 ^ 0 = 1`. 

In [17]:
def complement_build_mask(n: int) -> int:
    mask = 1
    while mask < n:
        mask = (mask << 1) | 1
    return mask ^ n

for n in [5, 7, 10, 0, 1, 8]:
    print(n, "->", complement_build_mask(n))

5 -> 2
7 -> 0
10 -> 5
0 -> 1
1 -> 0
8 -> 7


Vì sao điều kiện `mask < n`? Mask dạng `2^d - 1` là số lớn nhất có `d` bit. Ta dừng ở mask nhỏ nhất **≥ n** → đó là `2^d - 1` với `d` = số bit của `n`.

Cẩn thận: `n = 7 = 111`: `mask = 1 → 3 → 7`, dừng vì `7 < 7` sai → mask = 7 ✓. Nếu viết `mask <= n` thì mask chạy thành 15 → sai.

## 7. TRACE từng bước

### 7.1. Trace cách xây mask với `n = 10`

| Bước | mask (thập phân) | mask (nhị phân) | `mask < n`? | Hành động |
|:-:|:-:|:-:|:-:|---|
| 0 | 1 | `1` | 1 < 10 ✓ | `mask = (1<<1)\|1 = 3` |
| 1 | 3 | `11` | 3 < 10 ✓ | `mask = 7` |
| 2 | 7 | `111` | 7 < 10 ✓ | `mask = 15` |
| 3 | 15 | `1111` | 15 < 10 ✗ | dừng |

Kết quả: `15 ^ 10 = 1111 ^ 1010 = 0101 = 5`.

In [18]:
def trace_build_mask(n: int) -> int:
    mask = 1
    step = 0
    print(f"n = {n} ({bin(n)[2:]})")
    while mask < n:
        new_mask = (mask << 1) | 1
        print(f"  bước {step}: mask={mask:>3} ({bin(mask)[2:]:>5}) < n -> mask mới = {new_mask} ({bin(new_mask)[2:]})")
        mask = new_mask
        step += 1
    print(f"  dừng: mask={mask} ({bin(mask)[2:]}) >= n")
    res = mask ^ n
    width = len(bin(mask)) - 2
    print(f"  {bin(mask)[2:]} ^ {bin(n)[2:].zfill(width)} = {bin(res)[2:].zfill(width)} = {res}")
    return res

trace_build_mask(10)
print()
trace_build_mask(7)
print()
trace_build_mask(0)

n = 10 (1010)
  bước 0: mask=  1 (    1) < n -> mask mới = 3 (11)
  bước 1: mask=  3 (   11) < n -> mask mới = 7 (111)
  bước 2: mask=  7 (  111) < n -> mask mới = 15 (1111)
  dừng: mask=15 (1111) >= n
  1111 ^ 1010 = 0101 = 5

n = 7 (111)
  bước 0: mask=  1 (    1) < n -> mask mới = 3 (11)
  bước 1: mask=  3 (   11) < n -> mask mới = 7 (111)
  dừng: mask=7 (111) >= n
  111 ^ 111 = 000 = 0

n = 0 (0)
  dừng: mask=1 (1) >= n
  1 ^ 0 = 1 = 1


1

`zfill(width)` thêm số 0 bên trái cho đủ `width` ký tự — giúp các dòng nhị phân thẳng cột.

### 7.2. Trace cách lật từng bit (Cách 2) với `n = 10`

In [19]:
def trace_loop(n: int) -> int:
    if n == 0:
        print("n = 0 -> trả 1 (edge case)")
        return 1
    result, pos = 0, 0
    print(f"{'pos':>3} | {'n':>3} | bit | lật | result")
    while n > 0:
        bit = n % 2
        result += (1 - bit) << pos
        print(f"{pos:>3} | {n:>3} |  {bit}  |  {1 - bit}  | {result} ({bin(result)[2:]})")
        n //= 2
        pos += 1
    return result

print("Kết quả:", trace_loop(10))

pos |   n | bit | lật | result
  0 |  10 |  0  |  1  | 1 (1)
  1 |   5 |  1  |  0  | 1 (1)
  2 |   2 |  0  |  1  | 5 (101)
  3 |   1 |  1  |  0  | 5 (101)
Kết quả: 5


## 8. Code nộp LeetCode (Python 3)

In [20]:
class Solution:
    def bitwiseComplement(self, n: int) -> int:
        if n == 0:
            return 1
        mask = (1 << n.bit_length()) - 1
        return n ^ mask

sol = Solution()
assert sol.bitwiseComplement(5) == 2
assert sol.bitwiseComplement(7) == 0
assert sol.bitwiseComplement(10) == 5
assert sol.bitwiseComplement(0) == 1
print("Solution PASS ví dụ của đề + n = 0")

Solution PASS ví dụ của đề + n = 0


Phiên bản không dùng `bit_length` (dễ chuyển sang C++/Java):

In [21]:
class SolutionBuildMask:
    def bitwiseComplement(self, n: int) -> int:
        mask = 1
        while mask < n:
            mask = (mask << 1) | 1
        return mask ^ n

assert [SolutionBuildMask().bitwiseComplement(n) for n in [5, 7, 10, 0]] == [2, 0, 5, 1]
print("SolutionBuildMask PASS")

SolutionBuildMask PASS


## 9. Vì sao đúng?

Gọi `d` = số bit của `n` (với `n = 0` thì quy ước `d = 1`). Viết `n = b_{d-1} ... b_1 b_0`.

- `mask = 2^d - 1` có đúng `d` bit 1 ở các vị trí `0..d-1`, các vị trí cao hơn là 0.
- XOR từng bit: ở vị trí `i < d`, `b_i ^ 1 = 1 - b_i` (lật). Ở vị trí `i ≥ d`, `0 ^ 0 = 0` (không phát sinh bit thừa).
- Vậy `n ^ mask` đúng bằng số có các bit `1 - b_i`, tức complement theo định nghĩa.

**Liên hệ `mask - n`:** vì `n + complement` ở mỗi vị trí `i < d` là `b_i + (1 - b_i) = 1`, không có nhớ → tổng là `mask`. Nên `mask - n = mask ^ n`.

**Bất biến của vòng lặp xây mask:** trước mỗi lần kiểm tra, `mask = 2^t - 1` với `t ≥ 1`. Vòng dừng ở `t` nhỏ nhất thỏa `2^t - 1 ≥ n`, tức `t` = số bit của `n` (với `n ≥ 1`), hoặc `t = 1` khi `n ∈ {0, 1}`.

In [22]:
# Kiểm chứng tính chất n + complement = mask và mask - n == mask ^ n
for n in range(1, 50):
    mask = (1 << n.bit_length()) - 1
    assert n + complement_string(n) == mask
    assert mask - n == mask ^ n
print("Tính chất n + complement = mask đúng với n = 1..49")

Tính chất n + complement = mask đúng với n = 1..49


## 10. Phân tích Time / Space

Gọi `d = số bit của n` (`d ≤ 30` vì `n < 10^9 < 2^30 ≈ 1.07·10^9`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---|---|
| 1. Chuỗi | `bin` → lật ký tự → `int(s, 2)` | `O(d)` | `O(d)` |
| 2. Lật từng bit | `% 2`, `// 2`, cộng dồn | `O(d)` | `O(1)` |
| 3. `bit_length` + XOR | mask một phát | `O(1)`* | `O(1)` |
| 4. Tự xây mask | lặp thêm bit 1 | `O(d)` | `O(1)` |

\* `bit_length` và XOR trên số nhỏ (≤ 30 bit) là hằng số. Nói chặt thì phép toán số lớn tỉ lệ với số bit, nhưng với constraints này mọi cách đều rất nhanh.

## 11. Kiểm thử: mẫu đề, biên, đối chiếu

In [23]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS ({actual})")

cases = [
    ("đề 1: 5 = 101", 5, 2),
    ("đề 2: 7 = 111", 7, 0),
    ("đề 3: 10 = 1010", 10, 5),
    ("biên: 0", 0, 1),
    ("biên: 1", 1, 0),
    ("lũy thừa 2: 8 = 1000", 8, 7),
    ("2^k - 1: 1023", 1023, 0),
    ("gần max: 10^9 - 1", 10**9 - 1, (1 << 30) - 1 - (10**9 - 1)),
]
for name, n, exp in cases:
    check(name, Solution().bitwiseComplement(n), exp)

đề 1: 5 = 101                    PASS (2)
đề 2: 7 = 111                    PASS (0)
đề 3: 10 = 1010                  PASS (5)
biên: 0                          PASS (1)
biên: 1                          PASS (0)
lũy thừa 2: 8 = 1000             PASS (7)
2^k - 1: 1023                    PASS (0)
gần max: 10^9 - 1                PASS (73741824)


### 11.1. Đối chiếu mọi cách trên miền nhỏ và ngẫu nhiên có seed

In [24]:
import random
random.seed(1009)

solvers = {
    "string": complement_string,
    "string_join": complement_string_join,
    "loop": complement_loop,
    "mask": complement_mask,
    "build_mask": complement_build_mask,
    "Solution": Solution().bitwiseComplement,
    "SolutionString": SolutionString().bitwiseComplement,
}
inputs = list(range(0, 2049)) + [random.randint(0, 10**9 - 1) for _ in range(2000)]
for n in inputs:
    expected = complement_string(n)   # cách chuỗi làm oracle: bám sát định nghĩa
    for name, f in solvers.items():
        assert f(n) == expected, (name, n, f(n), expected)
print(f"{len(inputs)} input: mọi cách khớp nhau")

4049 input: mọi cách khớp nhau


### 11.2. Tính chất đáng kiểm: complement của complement

Complement **không** phải phép nghịch đảo hoàn hảo: `comp(comp(10)) = comp(5) = comp(101) = 010 = 2 ≠ 10` vì số 0 ở đầu bị bỏ. Đây là một tính chất dễ hiểu nhầm.

In [25]:
for n in [10, 5, 7, 6]:
    c1 = Solution().bitwiseComplement(n)
    c2 = Solution().bitwiseComplement(c1)
    print(f"n={n:>2} ({bin(n)[2:]:>4}) -> comp={c1} ({bin(c1)[2:]}) -> comp(comp)={c2} ({bin(c2)[2:]})")

n=10 (1010) -> comp=5 (101) -> comp(comp)=2 (10)
n= 5 ( 101) -> comp=2 (10) -> comp(comp)=1 (1)
n= 7 ( 111) -> comp=0 (0) -> comp(comp)=1 (1)
n= 6 ( 110) -> comp=1 (1) -> comp(comp)=0 (0)


## 12. Sai lầm người mới hay mắc

### 12.1. Dùng `~n`

In [26]:
def wrong_tilde(n):
    return ~n

print("sai :", wrong_tilde(5))
print("đúng:", Solution().bitwiseComplement(5))

sai : -6
đúng: 2


**Vì sao sai:** `~n` lật vô hạn bit (cả các bit 0 phía trước) → ra số âm `-n-1`. **Sửa:** XOR với mask chỉ gồm `d` bit 1.

### 12.2. Quên `n = 0` khi dùng `bit_length`

In [27]:
def wrong_zero(n):
    return n ^ ((1 << n.bit_length()) - 1)

print("sai :", wrong_zero(0))
print("đúng:", Solution().bitwiseComplement(0))

sai : 0
đúng: 1


**Vì sao sai:** `(0).bit_length() = 0` → mask 0. **Sửa:** `if n == 0: return 1` hoặc `max(1, n.bit_length())`.

### 12.3. Dùng `<=` khi xây mask

In [28]:
def wrong_le(n):
    mask = 1
    while mask <= n:
        mask = (mask << 1) | 1
    return mask ^ n

print("sai :", wrong_le(7), "| sai với 1:", wrong_le(1))
print("đúng:", Solution().bitwiseComplement(7), "|", Solution().bitwiseComplement(1))

sai : 8 | sai với 1: 2
đúng: 0 | 0


**Vì sao sai:** `n = 7 = 111` đã có mask đúng là 7, nhưng `7 <= 7` khiến mask thành 15 → `15 ^ 7 = 8`. **Sửa:** dùng `mask < n`.

### 12.4. Quên bỏ tiền tố `'0b'`

In [29]:
s = bin(5)
print("không bỏ tiền tố:", s, "-> lật thành", s.translate(str.maketrans("01", "10")))
try:
    int(s.translate(str.maketrans("01", "10")), 2)
except ValueError as e:
    print("ValueError:", e)

không bỏ tiền tố: 0b101 -> lật thành 1b010
ValueError: invalid literal for int() with base 2: '1b010'


**Vì sao sai:** `'0b101'` có chữ `'0'` ở tiền tố bị lật thành `'1'` → `'1b010'` không phải chuỗi nhị phân hợp lệ. **Sửa:** `bin(n)[2:]` hoặc `format(n, "b")`.

`try/except ValueError` bắt lỗi để notebook tiếp tục chạy — ta dùng nó để **minh họa** lỗi mà không làm dừng notebook.

## 13. Nhận diện pattern

Dấu hiệu:
- "lật bit", "complement", "đảo 0/1" → nghĩ tới **XOR với mask**.
- "toàn bit 1 có độ dài `d`" → `(1 << d) - 1`.
- "số bit của n" → `n.bit_length()` (nhớ ca `0`).

Bài liên quan:
- LeetCode 476 (Number Complement) — y hệt bài này (`n ≥ 1`).
- LeetCode 191 (Number of 1 Bits), 461 (Hamming Distance) — `&`, `^`, đếm bit.
- LeetCode 190 (Reverse Bits), 231 (Power of Two) — `n & (n - 1)`.
- LeetCode 1018 (Binary Prefix Divisible By 5) — đọc dần các bit `x = 2x + b`.

## 14. Biến thể

### 14.1. Biến thể A — lật bit trong khuôn **cố định** `w` bit

Ví dụ trong máy tính 8 bit: `5 = 00000101` → lật → `11111010 = 250`. Khi đó mask là `(1 << w) - 1`, không phụ thuộc `n`.

In [30]:
def complement_fixed_width(n: int, w: int) -> int:
    return n ^ ((1 << w) - 1)

print(complement_fixed_width(5, 8))     # 250
print(complement_fixed_width(0, 8))     # 255

250
255


### 14.2. Biến thể B — lật bit từ vị trí `l` đến `r` (tính từ phải, bắt đầu 0)

Mask chỉ có bit 1 ở đoạn `[l, r]`: `((1 << (r - l + 1)) - 1) << l`.

In [31]:
def flip_range(n: int, l: int, r: int) -> int:
    mask = ((1 << (r - l + 1)) - 1) << l
    return n ^ mask

n = 0b101101
print(bin(n), "-> lật bit 1..3 ->", bin(flip_range(n, 1, 3)))

0b101101 -> lật bit 1..3 -> 0b100011


## 15. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Tính complement của `n = 37` bằng tay: viết nhị phân, lật, đổi lại thập phân. Kiểm tra bằng công thức `mask - n`.

### Bài 2 — Dự đoán output
Đoán kết quả: `bin(12)`, `(12).bit_length()`, `12 ^ 15`, `~12`.

### Bài 3 — TODO: viết lại lời giải mask

In [32]:
def exercise_complement(n: int) -> int:
    # TODO 1: xử lý n == 0
    # TODO 2: tạo mask toàn bit 1 dài bằng n
    # TODO 3: XOR
    pass

# assert exercise_complement(5) == 2
# assert exercise_complement(0) == 1
# assert exercise_complement(8) == 7

### Bài 4 — TODO: đếm số bit 0 trong biểu diễn nhị phân của `n` (không có số 0 ở đầu)

Gợi ý liên hệ: số bit 0 của `n` = số bit 1 của complement.

In [33]:
def exercise_count_zero_bits(n: int) -> int:
    # TODO
    pass

# assert exercise_count_zero_bits(10) == 2   # 1010
# assert exercise_count_zero_bits(7) == 0
# assert exercise_count_zero_bits(0) == 1

### Bài 5 — TODO biến thể: trả về **chuỗi nhị phân** của complement có giữ số 0 ở đầu

Ví dụ `10 = "1010"` → `"0101"`. `7` → `"000"`.

In [34]:
def exercise_complement_str(n: int) -> str:
    # TODO
    pass

# assert exercise_complement_str(10) == "0101"
# assert exercise_complement_str(7) == "000"
# assert exercise_complement_str(0) == "1"

### Bài 6 — Mini challenge: complement **không dùng** `bit_length`, `bin`, chuỗi, và **không dùng vòng lặp xây mask từng bit**

Gợi ý: "lan" bit cao nhất xuống mọi bit thấp hơn bằng các phép `m |= m >> 1`, `m |= m >> 2`, `m |= m >> 4`, `m |= m >> 8`, `m |= m >> 16` (đủ cho số < 2^32).

In [35]:
def exercise_complement_smear(n: int) -> int:
    # TODO
    pass

# for n in range(0, 5000):
#     assert exercise_complement_smear(n) == Solution().bitwiseComplement(n)

## 16. Gợi ý ba mức

**Bài 1** — (1) chia 2 liên tiếp; (2) `37 = 32 + 4 + 1`; (3) `100101` → `011010`.

**Bài 2** — (1) `12 = 8 + 4`; (2) 4 bit; (3) `~x = -x - 1`.

**Bài 3** — (1) mục 6; (2) `n.bit_length()`; (3) `return n ^ ((1 << n.bit_length()) - 1)`.

**Bài 4** — (1) duyệt bit bằng `% 2`; (2) cẩn thận `n = 0`; (3) `bin(n)[2:].count("0")` là cách một dòng.

**Bài 5** — (1) làm việc với chuỗi; (2) `translate` hoặc `join`; (3) **không** gọi `int(...)` ở cuối.

**Bài 6** — (1) sau khi "lan", `m` thành `2^d - 1`; (2) bắt đầu `m = n`; (3) với `n = 0`, kết quả `m = 0` → cần `if n == 0: return 1`.

## 17. Đáp án tham khảo

**Bài 1:** `37 = 100101₂` → lật `011010₂ = 16 + 8 + 2 = 26`. Kiểm tra: `mask = 63`, `63 - 37 = 26` ✓.

**Bài 2:** `'0b1100'`, `4`, `12 ^ 15 = 3`, `~12 = -13`.

In [36]:
def answer_complement(n):
    if n == 0:
        return 1
    return n ^ ((1 << n.bit_length()) - 1)


def answer_count_zero_bits(n):
    if n == 0:
        return 1
    zeros = 0
    while n > 0:
        if n % 2 == 0:
            zeros += 1
        n //= 2
    return zeros


def answer_complement_str(n):
    return bin(n)[2:].translate(str.maketrans("01", "10"))


def answer_complement_smear(n):
    if n == 0:
        return 1
    m = n
    m |= m >> 1
    m |= m >> 2
    m |= m >> 4
    m |= m >> 8
    m |= m >> 16
    return m ^ n


assert answer_complement(37) == 26
assert (bin(12), (12).bit_length(), 12 ^ 15, ~12) == ("0b1100", 4, 3, -13)
assert answer_complement(5) == 2 and answer_complement(0) == 1 and answer_complement(8) == 7
assert answer_count_zero_bits(10) == 2 and answer_count_zero_bits(7) == 0 and answer_count_zero_bits(0) == 1
assert answer_complement_str(10) == "0101" and answer_complement_str(7) == "000" and answer_complement_str(0) == "1"
for n in range(0, 5000):
    assert answer_complement_smear(n) == Solution().bitwiseComplement(n)
    assert answer_count_zero_bits(n) == bin(n)[2:].count("0")
    assert answer_count_zero_bits(n) == bin(answer_complement(n)).count("1")
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 6:** giả sử bit cao nhất của `n` ở vị trí `d-1`. Sau `m |= m >> 1`, hai bit cao nhất là 1; sau `>> 2` là 4 bit; `>> 4` → 8; `>> 8` → 16; `>> 16` → 32 bit. Vì `n < 2^30`, mọi bit từ `d-1` xuống 0 đều thành 1 → `m = 2^d - 1` = mask. Kỹ thuật này gọi là **bit smearing**, dùng nhiều trong C/C++ để làm tròn lên lũy thừa của 2.

## 18. Tổng kết — template + checklist

**Template nhớ nhanh:**

```python
class Solution:
    def bitwiseComplement(self, n: int) -> int:
        if n == 0:
            return 1
        mask = (1 << n.bit_length()) - 1
        return n ^ mask
```

**Câu thần chú:** *"Số + phần bù = toàn bit 1 → phần bù = mask XOR số; nhớ số 0."*

**Checklist trước khi submit:**
- [ ] Đã xử lý `n = 0` → trả `1`?
- [ ] Không dùng `~n` trần (ra số âm)?
- [ ] Mask có **đúng** số bit của `n` (dùng `<` chứ không phải `<=` khi tự xây)?
- [ ] Nếu dùng chuỗi: đã bỏ `'0b'`?
- [ ] Đã test `1`, `7` (toàn bit 1), `8` (lũy thừa 2)?